In [0]:
from pyspark.sql import functions as F

CURRENT_TABLE = "sentinel_dev.silver.silver_orders_current"
HISTORY_TABLE = "sentinel_dev.silver.silver_orders_history"
QUARANTINE_TABLE = "sentinel_dev.silver.orders_quarantine"
VALIDATED_TABLE = "sentinel_dev.silver.silver_orders_validated"
current_df = spark.table(CURRENT_TABLE)
history_df = spark.table(HISTORY_TABLE)
quarantine_df = spark.table(QUARANTINE_TABLE)
validated_df = spark.table(VALIDATED_TABLE)

In [0]:
metrics = {
    "current_orders": current_df.count(),

    "historical_versions": history_df.count(),

    "validated_records": validated_df.count(),

    "quarantined_records": quarantine_df.count()
}

metrics

In [0]:
validated_count = validated_df.count()
quarantine_count = quarantine_df.count()

total_quality_records = validated_count + quarantine_count

quality_pass_rate = (
    validated_count / total_quality_records * 100
    if total_quality_records > 0
    else 0
)

print(f"Quality pass rate: {quality_pass_rate:.2f}%")

In [0]:
validated_count = validated_df.count()
quarantine_count = quarantine_df.count()

total_quality_records = validated_count + quarantine_count

quality_pass_rate = (
    validated_count / total_quality_records * 100
    if total_quality_records > 0
    else 0
)

print(f"Quality pass rate: {quality_pass_rate:.2f}%")

In [0]:
validated_count = validated_df.count()
quarantine_count = quarantine_df.count()

total_quality_records = validated_count + quarantine_count

quality_pass_rate = (
    validated_count / total_quality_records * 100
    if total_quality_records > 0
    else 0
)

print(f"Quality pass rate: {quality_pass_rate:.2f}%")

In [0]:
failed_rules_summary = (
    quarantine_df
        .select(
            F.explode("failed_rules").alias("failed_rule")
        )
        .groupBy("failed_rule")
        .count()
        .orderBy(F.col("count").desc())
)

display(failed_rules_summary)

In [0]:
freshness_df = (
    current_df
        .agg(
            F.max("ingested_at").alias("latest_ingestion"),
            F.max("order_timestamp_clean").alias("latest_business_event")
        )
)

display(freshness_df)

In [0]:
monitoring_row = (
    spark.createDataFrame(
        [(
            metrics["current_orders"],
            metrics["historical_versions"],
            metrics["validated_records"],
            metrics["quarantined_records"],
            quality_pass_rate
        )],
        """
        current_orders LONG,
        historical_versions LONG,
        validated_records LONG,
        quarantined_records LONG,
        quality_pass_rate DOUBLE
        """
    )
    .withColumn(
        "metric_timestamp",
        F.current_timestamp()
    )
)

In [0]:
(
    monitoring_row.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "sentinel_dev.monitoring.pipeline_health"
        )
)

In [0]:
display(
    spark.table(
        "sentinel_dev.monitoring.pipeline_health"
    )
    .orderBy(
        F.col("metric_timestamp").desc()
    )
)